# Bước 9 (bản 2) – Hướng 2: dùng một ít nhãn ở mạng mới, thêm "ngưỡng khớp tỷ lệ"

Lý thuyết từng bước, kèm ví dụ số, nằm trong tài liệu "TADP-IDS – Lý thuyết bước 9" Claude đã gửi (phần 7 là ngưỡng khớp tỷ lệ).

Bản 2 giữ nguyên mọi thứ của bản 1 và thêm một cách chọn ngưỡng mới cho M1, M2, M3 và mốc "đủ nhãn". Hai cách chạy song song, cùng mô hình, cùng điểm số, chỉ khác ngưỡng.

| Ô | Việc | Thời gian |
|---|---|---|
| 1 | Kiểm tra môi trường và mã nguồn | 1 phút |
| 2 | Huấn luyện 4 chuyên gia, rồi thử 3 cách dùng 100 / 500 / 1.000 / 5.000 nhãn trên 4 mạng mới | 15–30 phút |
| 3 | Tổng hợp | 1 phút |

**Không cần GPU (card đồ họa).** Kết quả lưu vào `MyDrive/TADP_Q1/results/b9_it_nhan_v2/` (thư mục mới, không đụng vào kết quả bản 1). Nếu Colab ngắt ở Ô 2, chạy lại Ô 2: mạng nào đã xong sẽ được bỏ qua.

## Ô 1 – Kiểm tra môi trường và mã nguồn

In [1]:
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
CODE_DIR = "/content/drive/MyDrive/TADP_Q1/code"
sys.path.insert(0, CODE_DIR)
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
try:
    import lightgbm, pyarrow
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "pyarrow"], check=True)
EXPECTED = {"__init__.py": 125, "cic_data.py": 8579, "config.py": 3263, "data.py": 9956, "experts.py": 5614,
            "few_label.py": 14033, "metrics.py": 4002, "preprocess.py": 3601, "shift.py": 3863}
ok = True
for f, size in EXPECTED.items():
    p = os.path.join(CODE_DIR, "tadp", f)
    real = os.path.getsize(p) if os.path.exists(p) else None
    if real != size:
        ok = False
        print(f"   SAI: {f} – mong đợi {size} byte, thực tế {real}")
print("Mã nguồn:", "ĐẦY ĐỦ, đúng phiên bản bước 9 bản 2" if ok else "CÓ LỖI – báo lại cho Claude")

Mounted at /content/drive
Mã nguồn: ĐẦY ĐỦ, đúng phiên bản bước 9 bản 2


## Ô 2 – Chạy thực nghiệm

In ra lần lượt:
- 4 dòng "chuyên gia …": mỗi mạng một chuyên gia LightGBM (cây tăng cường gradient), kèm ngưỡng nguồn;
- 4 dòng "[tên mạng] xong": mỗi mạng mới đã thử xong 3 cách (M1 chọn chuyên gia, M2 học mới, M3 kết hợp) ở 4 mức nhãn, mỗi mức bốc 3 lần, mỗi cách 2 kiểu ngưỡng.

Ô này đã tắt dòng cảnh báo "X does not have valid feature names" (cảnh báo vô hại của thư viện sklearn).

In [2]:
import sys, warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.data import load_all, make_splits, common_features
from tadp.few_label import run_few_label
B = "/content/drive/MyDrive/TADP_Q1"
DOMAINS_CIC = {"CIC-IDS2017": ["2017"], "CSE-CIC-IDS2018": ["2018"], "CIC-ToN-IoT": ["ton"], "CIC-UNSW-NB15": ["unsw"]}
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b9_it_nhan_v2",
                 DOMAINS=DOMAINS_CIC, SPLIT_MODE="episode", LGBM_ROUNDS=200)
data = load_all(cfg)
splits, _ = make_splits(cfg, data)
cols = common_features(data, "no_id")
run_few_label(cfg, data, splits, cols, pv="P1_log", budgets=(100, 500, 1000, 5000), repeats=(0, 1, 2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CIC-IDS2017: 999,712 dòng, 87 cột, tỷ lệ tấn công = 0.2411
CSE-CIC-IDS2018: 999,829 dòng, 87 cột, tỷ lệ tấn công = 0.0559
CIC-ToN-IoT: 1,000,028 dòng, 82 cột, tỷ lệ tấn công = 0.5302
CIC-UNSW-NB15: 1,000,399 dòng, 82 cột, tỷ lệ tấn công = 0.0251
chuyên gia CIC-IDS2017     : ngưỡng nguồn = 0.394 (27 giây)
chuyên gia CSE-CIC-IDS2018 : ngưỡng nguồn = 0.939 (25 giây)
chuyên gia CIC-ToN-IoT     : ngưỡng nguồn = 0.524 (21 giây)
chuyên gia CIC-UNSW-NB15   : ngưỡng nguồn = 0.033 (19 giây)
[CIC-IDS2017] xong trong 215 giây
[CSE-CIC-IDS2018] xong trong 221 giây
[CIC-ToN-IoT] xong trong 217 giây
[CIC-UNSW-NB15] xong trong 193 giây


,target,method,k,repeat,so_tan_cong_trong_k,ty_le_uoc_luong,nguong,macro_f1,auroc,fpr,macro_f1_constant
0,CIC-IDS2017,k0_trung_binh_deu,0,0,0,NaN,0.498396,0.445395,0.615349,0.001060,0.431467
1,CIC-IDS2017,k0_chon_theo_kiem_dinh_nguon,0,0,0,NaN,0.938566,0.638556,0.934467,0.000433,0.431467
2,CIC-IDS2017,k0_trong_so_TADP,0,0,0,NaN,0.845449,0.640083,0.822289,0.000472,0.431467
3,CIC-IDS2017,tham_khao_chuyen_gia_tot_nhat_biet_dap_an,0,0,0,NaN,0.002722,0.852720,0.934467,0.064412,0.431467
4,CIC-IDS2017,tran_du_nhan,200000,0,48335,NaN,0.393673,0.999576,0.999995,0.000032,0.431467
...,...,...,...,...,...,...,...,...,...,...,...
307,CIC-UNSW-NB15,M1_chon_chuyen_gia_khop_ty_le,5000,2,146,0.0292,0.986653,0.578333,0.628634,0.014042,0.493640
308,CIC-UNSW-NB15,M2_hoc_moi,5000,2,146,NaN,0.712341,0.836104,0.993605,0.008334,0.493640
309,CIC-UNSW-NB15,M2_hoc_moi_khop_ty_le,5000,2,146,0.0292,0.776104,0.829482,0.993605,0.007203,0.493640
310,CIC-UNSW-NB15,M3_ket_hop,5000,2,146,NaN,0.883368,0.704935,0.921270,0.013471,0.493640


## Ô 3 – Tổng hợp

- Bảng 9.1: Macro-F1 và AUROC trung bình 4 mạng mới, theo cách làm và số nhãn.
- Bảng 9.2: Macro-F1 theo từng mạng mới.
- Bảng 9.3: độ dao động giữa 3 lần bốc nhãn.
- Bảng 9.4 (mới): so sánh ngưỡng cũ và ngưỡng khớp tỷ lệ, cột "chenh_lech" = mới trừ cũ (dương là cách mới tốt hơn).

Chạy xong báo Claude.

In [3]:
import sys
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.few_label import summarize_few_label
B = "/content/drive/MyDrive/TADP_Q1"
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b9_it_nhan_v2")
summarize_few_label(cfg)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BẢNG 9.1 – Trung bình 4 mạng đích (mỗi mạng: trung bình 3 lần bốc nhãn)
                                                   macro_f1  auroc  macro_f1_constant
method                                    k                                         
M1_chon_chuyen_gia                        100        0.624  0.662              0.439
                                          500        0.628  0.672              0.439
                                          1000       0.631  0.668              0.439
                                          5000       0.641  0.666              0.439
M1_chon_chuyen_gia_khop_ty_le             100        0.603  0.662              0.439
                                          500        0.616  0.672              0.439
                                          1000       0.621  0.668              0.439
                                 

(                                                  macro_f1  auroc  macro_f1_constant
 method                                    k                                         
 M1_chon_chuyen_gia                        100        0.624  0.662              0.439
                                           500        0.628  0.672              0.439
                                           1000       0.631  0.668              0.439
                                           5000       0.641  0.666              0.439
 M1_chon_chuyen_gia_khop_ty_le             100        0.603  0.662              0.439
                                           500        0.616  0.672              0.439
                                           1000       0.621  0.668              0.439
                                           5000       0.631  0.666              0.439
 M2_hoc_moi                                100        0.657  0.805              0.439
                                           500        